In [28]:
!pip install gcsfs zarr regionmask

In [29]:
!pip install gcsfs zarr regionmask cftime

In [30]:
import pandas as pd, xarray as xr, gcsfs, regionmask, numpy as np

df = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")
fs = gcsfs.GCSFileSystem(token="anon")

In [31]:
scenarios = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
base = df.query("table_id == 'Amon' and variable_id == 'tas' and experiment_id in @scenarios")

counts = base.groupby(["source_id", "member_id"])["experiment_id"].nunique().reset_index()
complete = counts[counts.experiment_id == len(scenarios)]
print(complete.to_string())

            source_id   member_id  experiment_id
0          ACCESS-CM2    r1i1p1f1              5
1          ACCESS-CM2    r2i1p1f1              5
2          ACCESS-CM2    r3i1p1f1              5
3          ACCESS-CM2    r4i1p1f1              5
4          ACCESS-CM2    r5i1p1f1              5
28      ACCESS-ESM1-5   r31i1p1f1              5
29      ACCESS-ESM1-5   r32i1p1f1              5
31      ACCESS-ESM1-5   r34i1p1f1              5
32      ACCESS-ESM1-5   r35i1p1f1              5
33      ACCESS-ESM1-5   r36i1p1f1              5
34      ACCESS-ESM1-5   r37i1p1f1              5
35      ACCESS-ESM1-5   r38i1p1f1              5
36      ACCESS-ESM1-5   r39i1p1f1              5
45      AWI-CM-1-1-MR    r1i1p1f1              5
51        BCC-CSM2-MR    r1i1p1f1              5
57        CAMS-CSM1-0    r1i1p1f1              5
59        CAMS-CSM1-0    r2i1p1f1              5
60         CAS-ESM2-0    r1i1p1f1              5
62         CAS-ESM2-0    r3i1p1f1              5
64              CESM

In [32]:
picks = {"GFDL-ESM4": "r1i1p1f1"}  # add more after the first run works
regions = regionmask.defined_regions.ar6.land
rows = []

for model, member in picks.items():
    for scen in scenarios:
        sub = base.query("source_id == @model and member_id == @member and experiment_id == @scen")
        if sub.empty:
            print("missing", model, scen)
            continue
        print("processing", model, scen)
        ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
        annual = (ds.tas - 273.15).groupby("time.year").mean().load()

        mask = regions.mask_3D(ds)
        w = np.cos(np.deg2rad(ds.lat))
        reg = annual.weighted(mask * w).mean(dim=("lat", "lon"))

        out = reg.to_dataframe(name="tas").reset_index()[["year", "names", "tas"]]
        out["model"], out["scenario"] = model, scen
        rows.append(out)

data = pd.concat(rows).rename(columns={"names": "region"})

baseline = (data.query("scenario == 'historical' and 1850 <= year <= 1900")
                .groupby(["model", "region"]).tas.mean().rename("base"))
data = data.join(baseline, on=["model", "region"])
data["anomaly"] = data.tas - data.base

summary = (data.groupby(["region", "scenario", "year"]).anomaly
               .agg(median="median", low="min", high="max").reset_index())
summary[["median", "low", "high"]] = (summary.groupby(["region", "scenario"])[["median", "low", "high"]]
                                          .transform(lambda s: s.rolling(10, center=True, min_periods=1).mean()))
summary.round(3).to_csv("climate_paths.csv", index=False)

processing GFDL-ESM4 historical
processing GFDL-ESM4 ssp126
processing GFDL-ESM4 ssp245
processing GFDL-ESM4 ssp370
processing GFDL-ESM4 ssp585


In [33]:
print(summary.head())
print(summary.region.nunique(), "regions")
print(summary.query("year == 2100").groupby("scenario")["median"].mean())

              region    scenario  year    median       low      high
0  Arabian-Peninsula  historical  1850  0.039651  0.039651  0.039651
1  Arabian-Peninsula  historical  1851  0.012548  0.012548  0.012548
2  Arabian-Peninsula  historical  1852  0.078057  0.078057  0.078057
3  Arabian-Peninsula  historical  1853  0.076632  0.076632  0.076632
4  Arabian-Peninsula  historical  1854  0.032410  0.032410  0.032410
46 regions
scenario
ssp126    1.537331
ssp245    2.774913
ssp370    4.131084
ssp585    4.852788
Name: median, dtype: float64


In [34]:
preferred = ["GFDL-ESM4", "MPI-ESM1-2-LR", "IPSL-CM6A-LR", "MIROC6",
             "CanESM5", "MRI-ESM2-0", "ACCESS-CM2"]
avail = complete[complete.source_id.isin(preferred)].copy()
avail["is_r1"] = avail.member_id != "r1i1p1f1"   # False sorts first
avail = avail.sort_values(["is_r1", "member_id"]).drop_duplicates("source_id")
picks = dict(zip(avail.source_id, avail.member_id))
print(picks)

{'ACCESS-CM2': 'r1i1p1f1', 'CanESM5': 'r1i1p1f1', 'GFDL-ESM4': 'r1i1p1f1', 'IPSL-CM6A-LR': 'r1i1p1f1', 'MIROC6': 'r1i1p1f1', 'MPI-ESM1-2-LR': 'r1i1p1f1', 'MRI-ESM2-0': 'r1i1p1f1'}


In [35]:
print(summary.query("year == 2100 and region == 'Arabian-Peninsula'"))

                region scenario  year    median       low      high
250  Arabian-Peninsula   ssp126  2100  1.867773  1.867773  1.867773
336  Arabian-Peninsula   ssp245  2100  2.998468  2.998468  2.998468
422  Arabian-Peninsula   ssp370  2100  4.872388  4.872388  4.872388
508  Arabian-Peninsula   ssp585  2100  5.650703  5.650703  5.650703


In [36]:
print(data.model.unique())
print(data.groupby("model").scenario.unique())

['GFDL-ESM4']
model
GFDL-ESM4    [historical, ssp126, ssp245, ssp370, ssp585]
Name: scenario, dtype: object


In [37]:
print(summary.query("year == 2100 and region == 'Arabian-Peninsula'"))

                region scenario  year    median       low      high
250  Arabian-Peninsula   ssp126  2100  1.867773  1.867773  1.867773
336  Arabian-Peninsula   ssp245  2100  2.998468  2.998468  2.998468
422  Arabian-Peninsula   ssp370  2100  4.872388  4.872388  4.872388
508  Arabian-Peninsula   ssp585  2100  5.650703  5.650703  5.650703


In [38]:
print(picks)
print(complete.source_id.nunique(), "models have all 5 scenarios")
print(complete.source_id.unique()[:30])

{'ACCESS-CM2': 'r1i1p1f1', 'CanESM5': 'r1i1p1f1', 'GFDL-ESM4': 'r1i1p1f1', 'IPSL-CM6A-LR': 'r1i1p1f1', 'MIROC6': 'r1i1p1f1', 'MPI-ESM1-2-LR': 'r1i1p1f1', 'MRI-ESM2-0': 'r1i1p1f1'}
38 models have all 5 scenarios
['ACCESS-CM2' 'ACCESS-ESM1-5' 'AWI-CM-1-1-MR' 'BCC-CSM2-MR' 'CAMS-CSM1-0'
 'CAS-ESM2-0' 'CESM2' 'CESM2-WACCM' 'CMCC-CM2-SR5' 'CMCC-ESM2'
 'CNRM-CM6-1' 'CNRM-CM6-1-HR' 'CNRM-ESM2-1' 'CanESM5' 'CanESM5-CanOE'
 'EC-Earth3' 'EC-Earth3-Veg' 'EC-Earth3-Veg-LR' 'FGOALS-f3-L' 'FGOALS-g3'
 'GFDL-ESM4' 'GISS-E2-1-G' 'GISS-E2-1-H' 'IITM-ESM' 'INM-CM4-8'
 'INM-CM5-0' 'IPSL-CM6A-LR' 'KACE-1-0-G' 'MCM-UA-1-0' 'MIROC-ES2L']


In [39]:
print(data.model.unique())
print(summary.query("year == 2100 and region == 'Arabian-Peninsula'"))

['GFDL-ESM4']
                region scenario  year    median       low      high
250  Arabian-Peninsula   ssp126  2100  1.867773  1.867773  1.867773
336  Arabian-Peninsula   ssp245  2100  2.998468  2.998468  2.998468
422  Arabian-Peninsula   ssp370  2100  4.872388  4.872388  4.872388
508  Arabian-Peninsula   ssp585  2100  5.650703  5.650703  5.650703


In [40]:
# pick models (prefer r1i1p1f1)
preferred = ["GFDL-ESM4", "MPI-ESM1-2-LR", "IPSL-CM6A-LR", "MIROC6",
             "CanESM5", "MRI-ESM2-0", "ACCESS-CM2"]
avail = complete[complete.source_id.isin(preferred)].copy()
avail["is_r1"] = avail.member_id != "r1i1p1f1"
avail = avail.sort_values(["is_r1", "member_id"]).drop_duplicates("source_id")
picks = dict(zip(avail.source_id, avail.member_id))
print(picks)

regions = regionmask.defined_regions.ar6.land
rows = []

for model, member in picks.items():
    for scen in scenarios:
        sub = base.query("source_id == @model and member_id == @member and experiment_id == @scen")
        if sub.empty:
            print("missing", model, scen)
            continue
        print("processing", model, scen)
        try:
            ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
            annual = (ds.tas - 273.15).groupby("time.year").mean().load()

            mask = regions.mask_3D(ds)
            w = np.cos(np.deg2rad(ds.lat))
            reg = annual.weighted(mask * w).mean(dim=("lat", "lon"))

            out = reg.to_dataframe(name="tas").reset_index()[["year", "names", "tas"]]
            out["model"], out["scenario"] = model, scen
            rows.append(out)
        except Exception as e:
            print("  FAILED:", type(e).__name__, e)

data = pd.concat(rows).rename(columns={"names": "region"})
print("models in data:", data.model.unique())

baseline = (data.query("scenario == 'historical' and 1850 <= year <= 1900")
                .groupby(["model", "region"]).tas.mean().rename("base"))
data = data.join(baseline, on=["model", "region"])
data["anomaly"] = data.tas - data.base

summary = (data.groupby(["region", "scenario", "year"]).anomaly
               .agg(median="median", low="min", high="max").reset_index())
summary[["median", "low", "high"]] = (summary.groupby(["region", "scenario"])[["median", "low", "high"]]
                                          .transform(lambda s: s.rolling(10, center=True, min_periods=1).mean()))
summary.round(3).to_csv("climate_paths.csv", index=False)
print(summary.query("year == 2100 and region == 'Arabian-Peninsula'"))

{'ACCESS-CM2': 'r1i1p1f1', 'CanESM5': 'r1i1p1f1', 'GFDL-ESM4': 'r1i1p1f1', 'IPSL-CM6A-LR': 'r1i1p1f1', 'MIROC6': 'r1i1p1f1', 'MPI-ESM1-2-LR': 'r1i1p1f1', 'MRI-ESM2-0': 'r1i1p1f1'}
processing ACCESS-CM2 historical
processing ACCESS-CM2 ssp126


/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_

processing ACCESS-CM2 ssp245
processing ACCESS-CM2 ssp370
processing ACCESS-CM2 ssp585


/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)
/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_

processing CanESM5 historical
processing CanESM5 ssp126


/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)


processing CanESM5 ssp245
processing CanESM5 ssp370
processing CanESM5 ssp585


/tmp/ipykernel_7507/3497052349.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(fs.get_mapper(sub.zstore.iloc[0]), consolidated=True)


processing GFDL-ESM4 historical
processing GFDL-ESM4 ssp126
processing GFDL-ESM4 ssp245
processing GFDL-ESM4 ssp370
processing GFDL-ESM4 ssp585
processing IPSL-CM6A-LR historical
processing IPSL-CM6A-LR ssp126
processing IPSL-CM6A-LR ssp245
processing IPSL-CM6A-LR ssp370
processing IPSL-CM6A-LR ssp585
processing MIROC6 historical
processing MIROC6 ssp126
processing MIROC6 ssp245
processing MIROC6 ssp370
processing MIROC6 ssp585
processing MPI-ESM1-2-LR historical
processing MPI-ESM1-2-LR ssp126
processing MPI-ESM1-2-LR ssp245
processing MPI-ESM1-2-LR ssp370
processing MPI-ESM1-2-LR ssp585
processing MRI-ESM2-0 historical
processing MRI-ESM2-0 ssp126
processing MRI-ESM2-0 ssp245
processing MRI-ESM2-0 ssp370
processing MRI-ESM2-0 ssp585
models in data: ['ACCESS-CM2' 'CanESM5' 'GFDL-ESM4' 'IPSL-CM6A-LR' 'MIROC6'
 'MPI-ESM1-2-LR' 'MRI-ESM2-0']
                region scenario  year    median       low       high
250  Arabian-Peninsula   ssp126  2100  2.732065  2.166605   3.609420
536  Arabi